In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


In [8]:
df = playerScoring('Tyrese Maxey', s26, current_date, teamStarPlayer, projectedStartingFive)
len(df)

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:363: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])


136

### Load Player Data and Bookmaker Data

In [10]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_13589/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,John Collins,Over,14.0,-137,2025-11-21,2025-11-20T21:47:51Z
1,PrizePicks,player_points,John Collins,Under,14.0,-137,2025-11-21,2025-11-20T21:47:51Z
2,PrizePicks,player_points,James Harden,Over,27.5,-137,2025-11-21,2025-11-20T21:47:51Z
3,PrizePicks,player_points,James Harden,Under,27.5,-137,2025-11-21,2025-11-20T21:47:51Z
4,PrizePicks,player_points,Franz Wagner,Over,23.5,-137,2025-11-21,2025-11-20T21:47:51Z


### Update projected starting lineups

In [7]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 8 teams with confirmed lineups


### Top EVs for single bets

In [11]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
edge_threshold=0.30, stake=10, variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, 
max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 58 unique players...
Error getting prediction for Paul George: float division by zero


/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a cop

,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
0,Bobby Portis,FanDuel,14.5,9.42,Under,-104,1,5.41,0.562,High
1,Bobby Portis,BetRivers,13.5,9.42,Under,105,0,4.82,0.459,High
2,Bobby Portis,DraftKings,14.5,9.42,Under,-112,1,4.73,0.530,High
3,Bobby Portis,BetMGM,13.5,9.42,Under,100,0,4.47,0.447,High
4,Bobby Portis,BetRivers,14.5,9.42,Under,-120,1,4.36,0.523,High


## Top EVs for 2 leg bets

### Underdog picks

In [17]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['COMMENCE_TIME'] == '2025-11-21')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 37 players...
Error getting prediction for Paul George: float division by zero
Processing 32 players with valid predictions...
Generated 442 valid 2-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 30 combinations from 442 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Bobby Portis,Dominick Barlow,14.5,5.5,9.42,8.15,under,over,0,5.53,0.277,High,Med
1,Kobe Sanders,Bobby Portis,9.5,14.5,12.70,9.42,over,under,0,4.81,0.241,High,High
2,Tyus Jones,Dominick Barlow,3.5,5.5,4.46,8.15,over,over,0,3.84,0.192,Low,Med
3,Tyus Jones,Kobe Sanders,3.5,9.5,4.46,12.70,over,over,0,3.48,0.174,Low,High
4,Cam Spencer,VJ Edgecombe,12.5,14.5,9.37,17.21,under,over,0,1.68,0.084,High,High
5,Tristan da Silva,Cam Spencer,12.5,12.5,14.73,9.37,over,under,0,1.67,0.084,High,High
6,Zach Edey,VJ Edgecombe,12.5,14.5,14.88,17.21,over,over,0,1.65,0.082,Med,High
7,Tristan da Silva,Zach Edey,12.5,12.5,14.73,14.88,over,over,0,1.31,0.065,High,Med
8,Desmond Bane,Santi Aldama,21.5,16.5,19.40,13.80,under,under,0,0.61,0.031,High,High
9,Nickeil Alexander-Walker,Santi Aldama,17.5,16.5,19.65,13.80,over,under,0,0.55,0.027,High,High


### Prizepicks picks

In [18]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')& (dfsData['COMMENCE_TIME'] == '2025-11-21')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 55 players...
Error getting prediction for Paul George: float division by zero
Processing 49 players with valid predictions...
Generated 1046 valid 2-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 48 combinations from 1046 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Kobe Sanders,Bobby Portis,9.5,14.5,12.70,9.42,over,under,0,4.76,0.238,High,High
1,Tyus Jones,Bobby Portis,3.5,14.5,4.46,9.42,over,under,0,4.72,0.236,Low,High
2,Kobe Sanders,Tyus Jones,9.5,3.5,12.70,4.46,over,over,0,3.27,0.163,High,Low
3,Nicolas Batum,Goga Bitadze,5.0,4.5,6.35,5.94,over,over,0,2.66,0.133,Med,Low
4,Goga Bitadze,Zach Edey,4.5,12.5,5.94,14.88,over,over,0,2.52,0.126,Low,Med


## 3 leg parlay

### Underdog picks

In [19]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['COMMENCE_TIME'] == '2025-11-21')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 37 players...
Error getting prediction for Paul George: float division by zero
Processing 32 players with valid predictions...
Generated 4722 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 21 combinations from 4722 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Tyus Jones,Bobby Portis,Dominick Barlow,3.5,14.5,5.5,4.46,9.42,8.15,over,under,over,0,9.63,0.193,Low,High,Med
1,Kobe Sanders,Bobby Portis,Dominick Barlow,9.5,14.5,5.5,12.70,9.42,8.15,over,under,over,0,9.60,0.192,High,High,Med
2,Tyus Jones,Kobe Sanders,Cam Spencer,3.5,9.5,12.5,4.46,12.70,9.37,over,over,under,0,6.17,0.123,Low,High,High
3,Zach Edey,Cam Spencer,VJ Edgecombe,12.5,12.5,14.5,14.88,9.37,17.21,over,under,over,0,4.09,0.082,Med,High,High
4,Zach Edey,Santi Aldama,VJ Edgecombe,12.5,16.5,14.5,14.88,13.80,17.21,over,under,over,0,3.42,0.068,Med,High,High


### Prizepicks picks

In [20]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')& (dfsData['COMMENCE_TIME'] == '2025-11-21')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 55 players...
Error getting prediction for Paul George: float division by zero
Processing 49 players with valid predictions...
Generated 17411 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 32 combinations from 17411 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Kobe Sanders,Tyus Jones,Bobby Portis,9.5,3.5,14.5,12.70,4.46,9.42,over,over,under,0,8.76,0.175,High,Low,High
1,Goga Bitadze,Tyus Jones,Bobby Portis,4.5,3.5,14.5,5.94,4.46,9.42,over,over,under,0,8.53,0.171,Low,Low,High
2,Kobe Sanders,Goga Bitadze,Zach Edey,9.5,4.5,12.5,12.70,5.94,14.88,over,over,over,0,5.65,0.113,High,Low,Med
3,Brook Lopez,Nicolas Batum,Zach Edey,6.5,5.0,12.5,7.87,6.35,14.88,over,over,over,0,4.22,0.084,Med,Med,Med
4,Brook Lopez,Nicolas Batum,VJ Edgecombe,6.5,5.0,14.5,7.87,6.35,17.21,over,over,over,0,3.99,0.080,Med,Med,High
